# Video Subtitle Extractor on Runpod
Before running this notebook, expose HTTP port `7860` in the Pod configuration. Use an official PyTorch/Jupyter template and keep `/workspace` as the persistent volume mount.

In [ ]:
import os
import subprocess

subprocess.run(["nvidia-smi"], check=True)
print("Pod ID:", os.environ.get("RUNPOD_POD_ID", "not detected"))
print("CUDA version:", os.environ.get("CUDA_VERSION", "not detected"))

In [ ]:
from pathlib import Path

repo = Path("/workspace/video-subtitle-extractor")
remote = "https://github.com/lixinyiCQU/video-subtitle-extractor.git"
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", remote, str(repo)], check=True)
os.chdir(repo)
print("Repository:", repo)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg git
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install --upgrade -r requirements.txt

In [ ]:
from getpass import getpass

cache = Path("/workspace/hf-cache")
cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(cache)
os.environ["HF_HUB_CACHE"] = str(cache / "hub")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["GRADIO_AUTH_USERNAME"] = input("Gradio username [admin]: ").strip() or "admin"
password = getpass("Gradio password: ").strip()
if not password:
    raise ValueError("A non-empty Gradio password is required for the public Runpod proxy.")
os.environ["GRADIO_AUTH_PASSWORD"] = password
hf_token = getpass("HuggingFace token (optional, press Enter to skip): ").strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print("Model cache:", cache)

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")
if pod_id:
    print(f"Open: https://{pod_id}-7860.proxy.runpod.net")
else:
    print("Open the HTTP service for internal port 7860 from the Runpod Connect panel.")

In [ ]:
# Keep this cell running. Use the notebook interrupt button to stop the service.
!python gradio_app.py --server-name 0.0.0.0 --server-port 7860